In [ ]:
#Setup: no new library

In [1]:
import os
import sys
 
# run_query.py lives in week03/session_3_1/, next to this folder
sys.path.insert(0, os.path.abspath(os.path.join("..", "session_3_1")))
 
import duckdb
from run_query import connect_to_repo, run_sql
 
con = connect_to_repo()
print("DuckDB", duckdb.__version__)


DuckDB 2.0.0


In [ ]:
#Setup: the views, now with a real date

In [2]:
run_sql("""
CREATE OR REPLACE VIEW events AS
SELECT *,
       CAST(strptime(event_date, '%Y%m%d') AS DATE) AS report_date
FROM read_csv('data/raw/adverse_events_10k.csv',
              types = {'safety_report_id': 'VARCHAR',
                       'serious': 'VARCHAR',
                       'patient_age_unit': 'VARCHAR',
                       'event_date': 'VARCHAR'});
CREATE OR REPLACE VIEW drug_ref AS
SELECT * FROM read_csv('data/raw/drug_reference.csv');
SELECT event_date, report_date, typeof(report_date) AS report_date_type
FROM events
LIMIT 3;
""", con)

┌────────────┬─────────────┬──────────────────┐
│ event_date │ report_date │ report_date_type │
│  varchar   │    date     │     varchar      │
├────────────┼─────────────┼──────────────────┤
│ 20240701   │ 2024-07-01  │ DATE             │
│ 20240701   │ 2024-07-01  │ DATE             │
│ 20240701   │ 2024-07-01  │ DATE             │
└────────────┴─────────────┴──────────────────┘



In [ ]:
# 1. WITH: Give Each Step a Name

In [3]:
run_sql("""
CREATE OR REPLACE TABLE tests (student VARCHAR, test_no INTEGER,
                               score INTEGER);
INSERT INTO tests VALUES
    ('Asha', 1, 70),  ('Asha', 2, 78),  ('Asha', 3, 78),  ('Asha', 4, 90),
    ('Ravi', 1, 65),  ('Ravi', 2, 60),  ('Ravi', 3, 72),  ('Ravi', 4, 72),
    ('Meera', 1, 88), ('Meera', 2, 88), ('Meera', 3, 91), ('Meera', 4, 85);
SELECT student, COUNT(*) AS tests, AVG(score) AS avg_score
FROM tests
GROUP BY student
ORDER BY student;
""", con)

┌─────────┬───────┬───────────┐
│ student │ tests │ avg_score │
│ varchar │ int64 │  double   │
├─────────┼───────┼───────────┤
│ Asha    │     4 │      79.0 │
│ Meera   │     4 │      88.0 │
│ Ravi    │     4 │     67.25 │
└─────────┴───────┴───────────┘



In [4]:
run_sql("""
WITH student_avg AS (
    SELECT student, AVG(score) AS avg_score
    FROM tests
    GROUP BY student
)
SELECT student, avg_score
FROM student_avg
WHERE avg_score > 75
ORDER BY avg_score DESC, student;
""", con)

┌─────────┬───────────┐
│ student │ avg_score │
│ varchar │  double   │
├─────────┼───────────┤
│ Meera   │      88.0 │
│ Asha    │      79.0 │
└─────────┴───────────┘



In [5]:
run_sql("""
WITH drug_totals AS (
    SELECT drug_name, COUNT(*) AS reports
    FROM events
    GROUP BY drug_name
),
big_drugs AS (
    SELECT drug_name, reports
    FROM drug_totals
    WHERE reports >= 100
)
SELECT COUNT(*) AS big_drugs, SUM(reports) AS their_reports,
       (SELECT COUNT(*) FROM drug_totals) AS all_drugs
FROM big_drugs;
""", con)

┌───────────┬───────────────┬───────────┐
│ big_drugs │ their_reports │ all_drugs │
│   int64   │    int128     │   int64   │
├───────────┼───────────────┼───────────┤
│         4 │          1035 │      1125 │
└───────────┴───────────────┴───────────┘



In [ ]:
# 2. Your First Window Function: OVER ()

In [6]:
run_sql("""
SELECT student, score,
       round(AVG(score) OVER (), 1)          AS class_avg,
       round(score - AVG(score) OVER (), 1)  AS vs_class
FROM tests
WHERE test_no = 1
ORDER BY score DESC, student;
""", con)

┌─────────┬───────┬───────────┬──────────┐
│ student │ score │ class_avg │ vs_class │
│ varchar │ int32 │  double   │  double  │
├─────────┼───────┼───────────┼──────────┤
│ Meera   │    88 │      74.3 │     13.7 │
│ Asha    │    70 │      74.3 │     -4.3 │
│ Ravi    │    65 │      74.3 │     -9.3 │
└─────────┴───────┴───────────┴──────────┘



In [7]:
run_sql("""
WITH by_country AS (
    SELECT COALESCE(country, 'Not reported') AS country,
           COUNT(*) AS reports
    FROM events
    GROUP BY COALESCE(country, 'Not reported')
)
SELECT country, reports,
       SUM(reports) OVER ()                             AS all_reports,
       round(100.0 * reports / SUM(reports) OVER (), 1) AS pct_of_all
FROM by_country
ORDER BY reports DESC, country
LIMIT 6;
""", con)

┌──────────────┬─────────┬─────────────┬────────────┐
│   country    │ reports │ all_reports │ pct_of_all │
│   varchar    │  int64  │   int128    │   double   │
├──────────────┼─────────┼─────────────┼────────────┤
│ US           │    4234 │        6218 │       68.1 │
│ Not reported │     403 │        6218 │        6.5 │
│ GB           │     253 │        6218 │        4.1 │
│ EU           │     229 │        6218 │        3.7 │
│ JP           │     186 │        6218 │        3.0 │
│ CA           │     170 │        6218 │        2.7 │
└──────────────┴─────────┴─────────────┴────────────┘



In [ ]:
# 3. PARTITION BY: A Window for Each Group

In [8]:
run_sql("""
SELECT student, test_no, score,
       AVG(score) OVER (PARTITION BY student)          AS own_avg,
       score - AVG(score) OVER (PARTITION BY student)  AS vs_own_avg
FROM tests
ORDER BY student, test_no;
""", con)

┌─────────┬─────────┬───────┬─────────┬────────────┐
│ student │ test_no │ score │ own_avg │ vs_own_avg │
│ varchar │  int32  │ int32 │ double  │   double   │
├─────────┼─────────┼───────┼─────────┼────────────┤
│ Asha    │       1 │    70 │    79.0 │       -9.0 │
│ Asha    │       2 │    78 │    79.0 │       -1.0 │
│ Asha    │       3 │    78 │    79.0 │       -1.0 │
│ Asha    │       4 │    90 │    79.0 │       11.0 │
│ Meera   │       1 │    88 │    88.0 │        0.0 │
│ Meera   │       2 │    88 │    88.0 │        0.0 │
│ Meera   │       3 │    91 │    88.0 │        3.0 │
│ Meera   │       4 │    85 │    88.0 │       -3.0 │
│ Ravi    │       1 │    65 │   67.25 │      -2.25 │
│ Ravi    │       2 │    60 │   67.25 │      -7.25 │
│ Ravi    │       3 │    72 │   67.25 │       4.75 │
│ Ravi    │       4 │    72 │   67.25 │       4.75 │
└─────────┴─────────┴───────┴─────────┴────────────┘
  12 rows                                5 columns



In [9]:
run_sql("""
WITH drug_counts AS (
    SELECT r.drug_class, e.drug_name, COUNT(*) AS reports
    FROM events AS e
    INNER JOIN drug_ref AS r ON e.drug_name = r.drug_name
    GROUP BY r.drug_class, e.drug_name
)
SELECT drug_name, reports,
       SUM(reports) OVER (PARTITION BY drug_class)  AS class_reports,
       round(100.0 * reports
             / SUM(reports) OVER (PARTITION BY drug_class), 1)
                                                    AS pct_of_class
FROM drug_counts
WHERE drug_class = 'GLP-1 Receptor Agonist'
ORDER BY reports DESC, drug_name;
""", con)

┌─────────────┬─────────┬───────────────┬──────────────┐
│  drug_name  │ reports │ class_reports │ pct_of_class │
│   varchar   │  int64  │    int128     │    double    │
├─────────────┼─────────┼───────────────┼──────────────┤
│ VICTOZA     │      69 │           113 │         61.1 │
│ OZEMPIC     │      25 │           113 │         22.1 │
│ TRULICITY   │      14 │           113 │         12.4 │
│ LIRAGLUTIDE │       4 │           113 │          3.5 │
│ RYBELSUS    │       1 │           113 │          0.9 │
└─────────────┴─────────┴───────────────┴──────────────┘



In [ ]:
# 4. Ranking: ROW_NUMBER, RANK and DENSE_RANK

In [10]:
run_sql("""
SELECT student, test_no, score,
       ROW_NUMBER() OVER (PARTITION BY student
                          ORDER BY score DESC, test_no) AS row_num,
       RANK()       OVER (PARTITION BY student
                          ORDER BY score DESC)          AS rnk,
       DENSE_RANK() OVER (PARTITION BY student
                          ORDER BY score DESC)          AS dense_rnk
FROM tests
WHERE student IN ('Asha', 'Meera')
ORDER BY student, row_num;
""", con)

┌─────────┬─────────┬───────┬─────────┬───────┬───────────┐
│ student │ test_no │ score │ row_num │  rnk  │ dense_rnk │
│ varchar │  int32  │ int32 │  int64  │ int64 │   int64   │
├─────────┼─────────┼───────┼─────────┼───────┼───────────┤
│ Asha    │       4 │    90 │       1 │     1 │         1 │
│ Asha    │       2 │    78 │       2 │     2 │         2 │
│ Asha    │       3 │    78 │       3 │     2 │         2 │
│ Asha    │       1 │    70 │       4 │     4 │         3 │
│ Meera   │       3 │    91 │       1 │     1 │         1 │
│ Meera   │       1 │    88 │       2 │     2 │         2 │
│ Meera   │       2 │    88 │       3 │     2 │         2 │
│ Meera   │       4 │    85 │       4 │     4 │         3 │
└─────────┴─────────┴───────┴─────────┴───────┴───────────┘



In [ ]:
# Filtering on a rank: not in WHERE

In [11]:
try:
    run_sql("""
    SELECT student, test_no, score
    FROM tests
    WHERE RANK() OVER (PARTITION BY student ORDER BY score DESC) = 1;
    """, con)
except duckdb.BinderException as error:
    print(error)

Binder Error: WHERE clause cannot contain window functions!


In [12]:
run_sql("""
WITH ranked AS (
    SELECT student, test_no, score,
           RANK() OVER (PARTITION BY student ORDER BY score DESC) AS rnk
    FROM tests
)
SELECT student, test_no, score
FROM ranked
WHERE rnk = 1
ORDER BY student, test_no;
""", con)

┌─────────┬─────────┬───────┐
│ student │ test_no │ score │
│ varchar │  int32  │ int32 │
├─────────┼─────────┼───────┤
│ Asha    │       4 │    90 │
│ Meera   │       3 │    91 │
│ Ravi    │       3 │    72 │
│ Ravi    │       4 │    72 │
└─────────┴─────────┴───────┘



In [ ]:
# ROW_NUMBER for "the latest record of each"

In [13]:
run_sql("""
WITH numbered AS (
    SELECT drug_name, safety_report_id, event_date, reaction,
           ROW_NUMBER() OVER (PARTITION BY drug_name
                              ORDER BY event_date DESC,
                                       safety_report_id DESC) AS row_num
    FROM events
    WHERE drug_name IN ('DUPIXENT', 'SKYRIZI', 'RINVOQ')
)
SELECT drug_name, safety_report_id, event_date, reaction
FROM numbered
WHERE row_num = 1
ORDER BY drug_name;
""", con)

┌───────────┬──────────────────┬────────────┬──────────────────┐
│ drug_name │ safety_report_id │ event_date │     reaction     │
│  varchar  │     varchar      │  varchar   │     varchar      │
├───────────┼──────────────────┼────────────┼──────────────────┤
│ DUPIXENT  │ 26839509         │ 20260604   │ Blister          │
│ RINVOQ    │ 26839474         │ 20260604   │ Cardiac disorder │
│ SKYRIZI   │ 26839528         │ 20260604   │ Osteoarthritis   │
└───────────┴──────────────────┴────────────┴──────────────────┘



In [ ]:
# 5. LAG and LEAD: The Row Before and the Row After

In [14]:
run_sql("""
SELECT student, test_no, score,
       LAG(score)  OVER (PARTITION BY student ORDER BY test_no) AS prev,
       score - LAG(score) OVER (PARTITION BY student
                                ORDER BY test_no)            AS change,
       LEAD(score) OVER (PARTITION BY student ORDER BY test_no) AS next
FROM tests
WHERE student = 'Ravi'
ORDER BY test_no;
""", con)

┌─────────┬─────────┬───────┬───────┬────────┬───────┐
│ student │ test_no │ score │ prev  │ change │ next  │
│ varchar │  int32  │ int32 │ int32 │ int32  │ int32 │
├─────────┼─────────┼───────┼───────┼────────┼───────┤
│ Ravi    │       1 │    65 │  NULL │   NULL │    60 │
│ Ravi    │       2 │    60 │    65 │     -5 │    72 │
│ Ravi    │       3 │    72 │    60 │     12 │    72 │
│ Ravi    │       4 │    72 │    72 │      0 │  NULL │
└─────────┴─────────┴───────┴───────┴────────┴───────┘



In [15]:
run_sql("""
WITH monthly AS (
    SELECT CAST(date_trunc('month', report_date) AS DATE) AS month_start,
           COUNT(*) AS reports
    FROM events
    GROUP BY month_start
)
SELECT strftime(month_start, '%Y-%m') AS month, reports,
       LAG(reports) OVER (ORDER BY month_start) AS prev_month,
       round(100.0 * (reports - LAG(reports) OVER (ORDER BY month_start))
             / NULLIF(LAG(reports) OVER (ORDER BY month_start), 0), 1)
                                                AS pct_change
FROM monthly
ORDER BY month_start
LIMIT 6;
""", con)

┌─────────┬─────────┬────────────┬────────────┐
│  month  │ reports │ prev_month │ pct_change │
│ varchar │  int64  │   int64    │   double   │
├─────────┼─────────┼────────────┼────────────┤
│ 2024-07 │     260 │       NULL │       NULL │
│ 2024-08 │     252 │        260 │       -3.1 │
│ 2024-09 │     240 │        252 │       -4.8 │
│ 2024-10 │     273 │        240 │       13.8 │
│ 2024-11 │     260 │        273 │       -4.8 │
│ 2024-12 │     208 │        260 │      -20.0 │
└─────────┴─────────┴────────────┴────────────┘



In [ ]:
# The trap: LAG reads the previous row, not the previous period

In [16]:
run_sql("""
WITH cabo AS (
    SELECT CAST(date_trunc('quarter', report_date) AS DATE) AS qtr,
           COUNT(*) AS reports
    FROM events
    WHERE drug_name = 'CABOMETYX'
    GROUP BY qtr
)
SELECT qtr, reports,
       LAG(reports) OVER (ORDER BY qtr) AS prev_row
FROM cabo
ORDER BY qtr;
""", con)

┌────────────┬─────────┬──────────┐
│    qtr     │ reports │ prev_row │
│    date    │  int64  │  int64   │
├────────────┼─────────┼──────────┤
│ 2025-01-01 │      84 │     NULL │
│ 2025-07-01 │       1 │       84 │
│ 2026-01-01 │      85 │        1 │
└────────────┴─────────┴──────────┘



In [17]:
run_sql("""
WITH calendar AS (
    SELECT DISTINCT CAST(date_trunc('quarter', report_date) AS DATE) AS qtr
    FROM events
),
cabo AS (
    SELECT CAST(date_trunc('quarter', report_date) AS DATE) AS qtr,
           COUNT(*) AS reports
    FROM events
    WHERE drug_name = 'CABOMETYX'
    GROUP BY qtr
)
SELECT c.qtr, COALESCE(b.reports, 0) AS reports,
       LAG(COALESCE(b.reports, 0)) OVER (ORDER BY c.qtr) AS prev_quarter
FROM calendar AS c
LEFT JOIN cabo AS b ON b.qtr = c.qtr
ORDER BY c.qtr;
""", con)

┌────────────┬─────────┬──────────────┐
│    qtr     │ reports │ prev_quarter │
│    date    │  int64  │    int64     │
├────────────┼─────────┼──────────────┤
│ 2024-07-01 │       0 │         NULL │
│ 2024-10-01 │       0 │            0 │
│ 2025-01-01 │      84 │            0 │
│ 2025-04-01 │       0 │           84 │
│ 2025-07-01 │       1 │            0 │
│ 2025-10-01 │       0 │            1 │
│ 2026-01-01 │      85 │            0 │
│ 2026-04-01 │       0 │           85 │
└────────────┴─────────┴──────────────┘



In [ ]:
# 6. Frames: Which Rows the Window Uses

In [18]:
run_sql("""
SELECT test_no, score,
       SUM(score) OVER (ORDER BY test_no)        AS sum_no_frame,
       LAST_VALUE(score) OVER (ORDER BY test_no) AS last_no_frame
FROM tests
WHERE student = 'Asha'
ORDER BY test_no;
""", con)

┌─────────┬───────┬──────────────┬───────────────┐
│ test_no │ score │ sum_no_frame │ last_no_frame │
│  int32  │ int32 │    int128    │     int32     │
├─────────┼───────┼──────────────┼───────────────┤
│       1 │    70 │           70 │            70 │
│       2 │    78 │          148 │            78 │
│       3 │    78 │          226 │            78 │
│       4 │    90 │          316 │            90 │
└─────────┴───────┴──────────────┴───────────────┘



In [19]:
run_sql("""
SELECT test_no, score,
       FIRST_VALUE(score) OVER w  AS first_test,
       LAST_VALUE(score)  OVER w  AS last_test,
       NTH_VALUE(score, 2) OVER w AS second_test
FROM tests
WHERE student = 'Asha'
WINDOW w AS (ORDER BY test_no
             ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
ORDER BY test_no;
""", con)

┌─────────┬───────┬────────────┬───────────┬─────────────┐
│ test_no │ score │ first_test │ last_test │ second_test │
│  int32  │ int32 │   int32    │   int32   │    int32    │
├─────────┼───────┼────────────┼───────────┼─────────────┤
│       1 │    70 │         70 │        90 │          78 │
│       2 │    78 │         70 │        90 │          78 │
│       3 │    78 │         70 │        90 │          78 │
│       4 │    90 │         70 │        90 │          78 │
└─────────┴───────┴────────────┴───────────┴─────────────┘



In [ ]:
# Ties in a running total

In [20]:
run_sql("""
SELECT test_no, score,
       SUM(score) OVER (ORDER BY score)            AS range_default,
       SUM(score) OVER (ORDER BY score, test_no
                        ROWS BETWEEN UNBOUNDED PRECEDING
                                 AND CURRENT ROW)  AS rows_frame
FROM tests
WHERE student = 'Asha'
ORDER BY score, test_no;
""", con)

┌─────────┬───────┬───────────────┬────────────┐
│ test_no │ score │ range_default │ rows_frame │
│  int32  │ int32 │    int128     │   int128   │
├─────────┼───────┼───────────────┼────────────┤
│       1 │    70 │            70 │         70 │
│       2 │    78 │           226 │        148 │
│       3 │    78 │           226 │        226 │
│       4 │    90 │           316 │        316 │
└─────────┴───────┴───────────────┴────────────┘



In [ ]:
# 7. Running Totals and Moving Averages

In [21]:
run_sql("""
WITH monthly AS (
    SELECT CAST(date_trunc('month', report_date) AS DATE) AS month_start,
           COUNT(*) AS reports
    FROM events
    WHERE drug_name = 'DUPIXENT'
    GROUP BY month_start
)
SELECT strftime(month_start, '%Y-%m') AS month, reports,
       SUM(reports) OVER (ORDER BY month_start
                          ROWS BETWEEN UNBOUNDED PRECEDING
                                   AND CURRENT ROW)   AS so_far,
       COUNT(*) OVER w3                               AS months_in_w,
       CASE WHEN COUNT(*) OVER w3 = 3
            THEN round(AVG(reports) OVER w3, 1) END   AS avg_3
FROM monthly
WINDOW w3 AS (ORDER BY month_start
              ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
ORDER BY month_start
LIMIT 8;
""", con)

┌─────────┬─────────┬────────┬─────────────┬────────┐
│  month  │ reports │ so_far │ months_in_w │ avg_3  │
│ varchar │  int64  │ int128 │    int64    │ double │
├─────────┼─────────┼────────┼─────────────┼────────┤
│ 2024-07 │       8 │      8 │           1 │   NULL │
│ 2024-08 │       4 │     12 │           2 │   NULL │
│ 2024-09 │       9 │     21 │           3 │    7.0 │
│ 2024-10 │      53 │     74 │           3 │   22.0 │
│ 2024-12 │      35 │    109 │           3 │   32.3 │
│ 2025-01 │      20 │    129 │           3 │   36.0 │
│ 2025-02 │       4 │    133 │           3 │   19.7 │
│ 2025-04 │      10 │    143 │           3 │   11.3 │
└─────────┴─────────┴────────┴─────────────┴────────┘



In [ ]:
# 8. From SQL to pandas and Back

In [22]:
monthly = con.sql("""
SELECT CAST(date_trunc('month', report_date) AS DATE) AS month_start,
       COUNT(*) AS reports
FROM events
GROUP BY month_start
ORDER BY month_start
""").df()
print(type(monthly).__name__, monthly.shape)
print(monthly.dtypes.to_string())

DataFrame (24, 2)
month_start    datetime64[us]
reports                 int64


In [23]:
monthly["pandas_pct"] = (monthly["reports"].pct_change() * 100).round(1)
 
sql_side = con.sql("""
WITH monthly AS (
    SELECT CAST(date_trunc('month', report_date) AS DATE) AS month_start,
           COUNT(*) AS reports
    FROM events
    GROUP BY month_start
)
SELECT month_start,
       round(100.0 * (reports - LAG(reports) OVER (ORDER BY month_start))
             / LAG(reports) OVER (ORDER BY month_start), 1) AS sql_pct
FROM monthly
ORDER BY month_start
""").df()
 
both = monthly.merge(sql_side, on="month_start", how="left",
                     validate="one_to_one")
exact = (both["pandas_pct"] == both["sql_pct"]) | (
    both["pandas_pct"].isna() & both["sql_pct"].isna())
print(both[~exact].to_string(index=False))
print(f"{int(exact.sum())} of {len(both)} months are identical")

month_start  reports  pandas_pct  sql_pct
 2024-10-01      273        13.7     13.8
23 of 24 months are identical


In [ ]:
# 9. Putting It Together: The Lab